In [ ]:
""" import os
import random
import shutil

# Define the paths
dataset_path = "Real and Fake Face Detection Dataset/"
train_path = os.path.join(dataset_path, "train")
test_path = os.path.join(dataset_path, "test")

categories = ["Fake", "Real"]

# Create train and test directories if they don't exist
os.makedirs(train_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)

# Create train and test subdirectories for each category
for category in categories:
    train_category_path = os.path.join(train_path, category)
    test_category_path = os.path.join(test_path, category)
    os.makedirs(train_category_path, exist_ok=True)
    os.makedirs(test_category_path, exist_ok=True)

# Define the split ratio (0.1 for 10%)
split_ratio = 0.1

# Iterate through each category
for category in categories:
    category_path = os.path.join(dataset_path, category)
    images = os.listdir(category_path)
    
    # Calculate the number of images for testing
    num_test_images = int(len(images) * split_ratio)
    
    # Randomly select images for testing
    test_images = random.sample(images, num_test_images)
    
    # Move test images to the test directory
    for img in test_images:
        src = os.path.join(category_path, img)
        dst = os.path.join(test_path, category, img)
        shutil.move(src, dst)
    
    # Move the remaining images to the train directory
    remaining_images = [img for img in images if img not in test_images]
    for img in remaining_images:
        src = os.path.join(category_path, img)
        dst = os.path.join(train_path, category, img)
        shutil.move(src, dst)
 """

In [ ]:
BATCH_SIZE=32
IMAGE_SIZE=(600,600)

In [ ]:
train_data_dir = "Real and Fake Face Detection Dataset/train"
test_data_dir = "Real and Fake Face Detection Dataset/test"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    subset='training',
    validation_split=0.15,
    seed=42
)

validation_data = tf.keras.utils.image_dataset_from_directory(
    train_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    subset='validation',
    validation_split=0.15,
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
)

In [ ]:
class_names=train_data.class_names
class_names

In [ ]:
for image_batch,label_batch in train_data.take(1):
    print(image_batch.shape)
    print(label_batch.shape)

In [ ]:
plt.figure(figsize=(10,4))
for image,label in train_data.take(1):
    for i in range(10):
        ax = plt.subplot(2,5,i+1)
        plt.imshow(image[i].numpy().astype('uint8'))
        plt.title(class_names[label[i]])
        plt.axis('off')

In [ ]:
train_data = train_data.map(lambda x,y:(x/255,y))
validation_data_data = validation_data.map(lambda x,y:(x/255,y))
test_data = test_data.map(lambda x,y:(x/255,y))

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(600, 600, 3)))
model.add(data_augmentation)

model.add(tf.keras.layers.Conv2D(32,kernel_size=3,activation='relu'))
model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Conv2D(64,kernel_size=3,activation='relu'))
model.add(tf.keras.layers.MaxPooling2D())

# model.add(tf.keras.layers.Conv2D(128,kernel_size=3,activation='relu'))
# model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.BatchNormalization())

model.add(tf.keras.layers.Flatten())

model.add(tf.keras.layers.Dense(128,activation='relu'))
# model.add(tf.keras.layers.Dense(128,activation='relu'))
model.add(tf.keras.layers.Dense(32,activation='relu'))

model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

In [ ]:
history = model.fit(train_data,
                    epochs=10,
                    validation_data=validation_data)